# Implementasi Bab 11 — PCA, t-SNE, dan LDA

Reduksi dimensi dataset Digits, visualisasi tiga metode, dan evaluasi kompresi PCA terhadap klasifikasi [ML-MODUL-01, hlm. 74–75].


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Dimensi asli:", X.shape)


In [ ]:
pca = PCA(n_components=0.90, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled)
X_tsne = TSNE(
    n_components=2, perplexity=30, random_state=SEED, init="pca", learning_rate="auto"
).fit_transform(X_scaled)
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)
print("Komponen untuk 90% variance:", pca.n_components_)
print("Total variance:", round(pca.explained_variance_ratio_.sum(), 3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, embedding, title in zip(
    axes,
    [pca_2d, X_tsne, X_lda],
    ["PCA 2D", "t-SNE 2D", "LDA 2D"],
):
    scatter = axis.scatter(embedding[:, 0], embedding[:, 1], c=y, cmap="tab10", s=10)
    axis.set_title(title)
fig.colorbar(scatter, ax=axes, label="Digit")
plt.show()


In [ ]:
cumulative_variance = np.cumsum(PCA(random_state=SEED).fit(X_scaled).explained_variance_ratio_)
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(0.90, color="red", linestyle="--", label="90%")
plt.xlabel("Jumlah komponen")
plt.ylabel("Cumulative explained variance")
plt.title("Scree/Cumulative Variance Plot")
plt.legend()
plt.show()


## Latihan: dampak kompresi PCA

Split dilakukan sebelum fitting scaler dan PCA agar test set tidak memengaruhi transformasi [ML-MODUL-01, hlm. 75].


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
compression_cv_rows = []
compression_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for n_components in [0.80, 0.90, 0.95, None]:
    compression_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=SEED)),
        ("model", LogisticRegression(max_iter=3000, random_state=SEED)),
    ])
    cv_accuracy = cross_val_score(
        compression_pipeline, X_train_raw, y_train, cv=compression_cv, scoring="accuracy"
    ).mean()
    compression_cv_rows.append((n_components, cv_accuracy))
compression_frame = pd.DataFrame(
    compression_cv_rows, columns=["variance_target", "cv_accuracy"]
)
display(compression_frame)
selected_components = compression_frame.sort_values("cv_accuracy", ascending=False).iloc[0]["variance_target"]
if pd.isna(selected_components):
    selected_components = None
best_compression_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=selected_components, random_state=SEED)),
    ("model", LogisticRegression(max_iter=3000, random_state=SEED)),
]).fit(X_train_raw, y_train)
final_test_accuracy = accuracy_score(y_test, best_compression_pipeline.predict(X_test_raw))
selected_dimensions = best_compression_pipeline.named_steps["pca"].n_components_
print("Konfigurasi PCA terpilih:", selected_components)
print("Jumlah dimensi:", selected_dimensions)
print("Akurasi test final:", round(final_test_accuracy, 3))


## Kesimpulan

PCA bersifat linear dan dapat mentransformasi data baru; t-SNE unggul untuk eksplorasi visual lokal tetapi bukan pipeline transformasi data baru; LDA memakai label untuk mencari proyeksi diskriminatif [ML-MODUL-01, hlm. 75].
